# Toy example: from ECMs to community composition

A minimal example of the calculation of community modes from the elementary conversion modes (ECMs) of individual species. 
The ECMs of each species and slack exchange reactions are placed into one matrix, community modes are calculated with efmtool, and community composition calculated from the relative contributions of the ECMs.

In [2]:
import efmtool
import numpy as np
import pandas as pd

options = efmtool.get_default_options()
options["level"] = "WARNING"

## Example 1: one ECM per species

Species 1 takes up A and secretes B; species 2 takes up B and secretes C.

Placing the two ECMs side by side as columns is not enough: the system would not be at steady state, since A and C have nowhere to come from or go to, and B produced by species 1 has nowhere to go unless it is either secreted or consumed by species 2. Three community exchange reactions fix this - `SA`, `SB`, `SC` - one per metabolite exchanged with the environment.

`SA` is reversible (`revs = 1`) because A can be taken up from the medium. B and C are not present in the medium, so they can only be secreted (`revs = 0`, irreversible).

In [3]:
matrix = np.array([[-1, 0, -1, 0, 0],
                    [1, -1, 0, -1, 0],
                    [0, 1, 0, 0, -1]])
revs = [0, 0, 1, 0, 0]
mets = ["A", "B", "C"]
rxns = ["ECM1", "ECM2", "SA", "SB", "SC"]

In [4]:
cefms = efmtool.calculate_efms(matrix, revs, rxns, mets, options)
cefms = pd.DataFrame(cefms, index=rxns)
cefms

,0,1
ECM1,1.0,1.0
ECM2,0.0,1.0
SA,-1.0,-1.0
SB,1.0,0.0
SC,0.0,1.0


Two community modes come out: one where only species 1 is present (A converted to B), and one where both are present and the community converts A all the way to C.

To turn these into relative abundances, sum each species' own ECM rows (its contribution across all of its own ECMs) and normalize so the two species sum to 1. Species are told apart by their row-name prefix (`ECM1*` vs `ECM2*`) - the same convention `analyze_ecms.ipynb` uses.

In [5]:
def compute_abundances(efms):
    species1 = efms[efms.index.str.startswith("ECM1")].sum(axis=0)
    species2 = efms[efms.index.str.startswith("ECM2")].sum(axis=0)
    abundances = pd.DataFrame({"species1": species1, "species2": species2})
    return abundances.div(abundances.sum(axis=1), axis=0)


compute_abundances(cefms)

,species1,species2
0,1.0,0.0
1,0.5,0.5


As expected: the species-1-only mode is 100% species 1, and the cross-feeding mode is a 50/50 community.

## Example 2: species 1 has two ECMs

Now species 1 can run two different conversions: ECM11 (A -> B) and ECM12 (A -> 0.5 B + 0.5 C). Species 2 still runs B -> C.

In [6]:
matrix = np.array([[-1, -1, 0, -1, 0, 0],
                    [1, 0.5, -1, 0, -1, 0],
                    [0, 0.5, 1, 0, 0, -1]])
revs = [0, 0, 0, 1, 0, 0]
mets = ["A", "B", "C"]
rxns = ["ECM11", "ECM12", "ECM2", "SA", "SB", "SC"]

In [7]:
efms = efmtool.calculate_efms(matrix, revs, rxns, mets, options)
efms = pd.DataFrame(efms, index=rxns)
efms

,0,1,2,3
ECM11,0.0,0.0,1.0,1.0
ECM12,2.0,2.0,0.0,0.0
ECM2,1.0,0.0,0.0,1.0
SA,-2.0,-2.0,-1.0,-1.0
SB,0.0,1.0,1.0,0.0
SC,2.0,1.0,0.0,1.0


Four community modes this time. `compute_abundances` sums `ECM11` and `ECM12` together for species 1's total contribution, so it works unchanged even though species 1 now has two ECMs instead of one.

In [8]:
compute_abundances(efms)

,species1,species2
0,0.666667,0.333333
1,1.000000,0.000000
2,1.000000,0.000000
3,0.500000,0.500000


* Mode 0: a two-thirds/one-third community, species 1 running ECM12 and species 2 consuming the B it doesn't already convert to C itself.
* Mode 1: species 1 alone, running ECM12. 
* Mode 2: species 1 alone, running ECM11 (which already makes some C by itself, but no species 2 is present).
* Mode 3: a 50/50 community, species 1 running ECM11 and species 2 consuming its B.

(Note that the order of the computed modes might be different on different machines).